1. Importing Necessary Libraries

In [13]:
from pathlib import Path
import os, re, time, json
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Dict, Tuple, Union
import cv2
import tensorflow as tf
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, roc_curve,
    confusion_matrix, classification_report
)

2. Configuration — Paths & Datasets

In [14]:
# Project paths 

CSV_PATH    = Path("../../annotations/new_annotations.csv").resolve()
IMAGES_ROOT = Path("../../datasets").resolve()
MODELS_DIR  = Path("../../models").resolve()
ANNOTATIONS_DIR = Path("../../annotations/adv_annotations").resolve()
REL_ROOT    = Path("../../").resolve()

# Model file paths
ANN_PATH = MODELS_DIR / "traffic_light_ann.keras"
CNN_PATH = MODELS_DIR / "traffic_light_cnn.keras"
RNN_PATH = MODELS_DIR / "traffic_light_rnn.keras"

# Dataset definitions 
# clean_test refer to original clean splits in CSV
# Annotations_* refer to adversarial attack CSV files
DATASETS = {
    "clean_train": CSV_PATH,
    "clean_test": CSV_PATH,
    "Annotations_ANN_fgsm": ANNOTATIONS_DIR / "Annotations_ANN_fgsm.csv",
    "Annotations_ANN_pgd":  ANNOTATIONS_DIR / "Annotations_ANN_pgd.csv",
    "Annotations_CNN_fgsm": ANNOTATIONS_DIR / "Annotations_CNN_fgsm.csv",
    "Annotations_CNN_pgd":  ANNOTATIONS_DIR / "Annotations_CNN_pgd.csv",
    "Annotations_RNN_fgsm":  ANNOTATIONS_DIR / "Annotations_RNN_fgsm.csv",
    "Annotations_RNN_pgd":  ANNOTATIONS_DIR / "Annotations_RNN_pgd.csv",
}

# Image/sequence sizes
IMG_H, IMG_W, IMG_C = 50, 50, 3

# Evaluation settings
SPLIT_FOR_CLEAN = {
    "clean_test": "test",
    "clean_train": "train",
}    # CSV column "split" mapping
BATCH_SIZE   = 512      # batch size during evaluation
NUM_BINS_ECE = 15       # number of bins for Expected Calibration Error (ECE)

# Preprocessing & input binding
COLOR_SPACE = "BGR"    # choose 'BGR' (OpenCV default) or 'RGB' (if converted during training)
BIND_BY_NAME = True    # True: feed dict by input tensor names; False: feed by order of model.inputs

# Output directory for results (Excel + plots)
OUT_DIR = Path("./")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_XLSX = OUT_DIR / "evaluation_summary.xlsx"

3. Utility Functions: Data Loading, Preprocessing, Metrics, and Input Binding

In [15]:
# Class names (fixed order: 0=go, 1=warning, 2=stop) 
CLASS_NAMES = ['go','warning','stop']  # 0,1,2

# Check if a path exists
def check_exists(path: Path):
    """Raise FileNotFoundError if the given path does not exist."""
    if not path.exists():
        raise FileNotFoundError(f"Path not found: {path}")
    return path

# Ensure all model files are present
for p in [ANN_PATH, CNN_PATH, RNN_PATH]:
    check_exists(p)

# Load CSV with flexible separator and harmonized column names 
def load_csv_safely(path: Path) -> pd.DataFrame:
    """Load CSV (comma or semicolon) and normalize column names like image_id, label."""
    with open(path, 'r', encoding='utf-8', errors='ignore') as f:
        head = f.read(4096)
    sep = ';' if head.count(';') > head.count(',') else ','
    df = pd.read_csv(path, sep=sep, engine='python')
    df.columns = [c.strip() for c in df.columns]

    # Normalize variant column names
    rename = {}
    for c in df.columns:
        lc = c.strip().lower()
        if lc in ('imageid', 'image_id '): rename[c] = 'image_id'
        if lc == 'label ': rename[c] = 'label'
    if rename:
        df = df.rename(columns=rename)
    return df

# Unified label normalization 
LABEL_MAP_STR = {'go':0, 'warning':1, 'stop':2}

def normalize_labels(series: pd.Series) -> np.ndarray:
    """
    Normalize labels from string ('go','warning','stop') or numeric (0-2 or 1-3).
    Returns numpy array of int labels in {0,1,2}.
    """
    if series.dtype == object:
        s = series.astype(str).str.strip().str.lower().map(LABEL_MAP_STR)
        if s.isna().any():
            bad = series[s.isna()].unique()[:5]
            raise ValueError(f"Unrecognized labels: {bad}")
        return s.values.astype(int)
    vals = series.astype(int).values
    if vals.min() >= 1 and vals.max() <= 3: return vals - 1
    if vals.min() >= 0 and vals.max() <= 2: return vals
    raise ValueError("Unexpected numeric label range; expected {0,1,2} or {1,2,3}.")

# Resolve image path (handles relative/absolute and nested adv paths) 
from pathlib import Path

def resolve_image_path(p):
    p = str(p).strip().replace("\\", "/")

    if not p.startswith("../../datasets/"):
        p = "../../datasets/" + p

    path = Path(p)

    if path.exists():
        return path.resolve()

    raise FileNotFoundError(f"Cannot resolve image path: {p}")

# Read and preprocess image 
def read_image_tensor(path: Path) -> np.ndarray:
    """
    Load image from disk, convert color if needed, resize to (IMG_W, IMG_H),
    and normalize to [0,1].
    Works better on Windows paths with non-ASCII characters.
    """
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Image not found: {path}")

    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_COLOR)

    if img is None:
        raise OSError(f"Failed to read image: {path}")

    if COLOR_SPACE == "RGB":
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = cv2.resize(img, (IMG_W, IMG_H), interpolation=cv2.INTER_AREA)
    return img.astype(np.float32) / 255.0

# Expected Calibration Error (ECE) 
def ece_top1(probs: np.ndarray, y_true: np.ndarray, n_bins: int = 15) -> float:
    """Compute Expected Calibration Error (ECE) for top-1 predictions."""
    conf = probs.max(axis=1)
    preds = probs.argmax(axis=1)
    correct = (preds == y_true).astype(np.float32)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    inds = np.digitize(conf, bins) - 1
    ece = 0.0
    for b in range(n_bins):
        m = inds == b
        if not np.any(m): continue
        ece += (m.mean()) * abs(correct[m].mean() - conf[m].mean())
    return float(ece)

# One-vs-Rest ROC metrics (AUROC, AUPRC, TPR at FPR=1%) 
def roc_ovr_metrics(y_true: np.ndarray, probs: np.ndarray) -> Tuple[float,float,float]:
    """
    Compute macro AUROC, macro AUPRC, and mean TPR at FPR=1% across classes.
    """
    K = probs.shape[1]
    yb = label_binarize(y_true, classes=list(range(K)))
    aucs, aups, tprs = [], [], []
    for k in range(K):
        yk = yb[:,k]; sk = probs[:,k]
        if yk.sum()==0 or yk.sum()==len(yk): continue
        aucs.append(roc_auc_score(yk, sk))
        from sklearn.metrics import average_precision_score
        aups.append(average_precision_score(yk, sk))
        fpr, tpr, _ = roc_curve(yk, sk); target=0.01
        import numpy as np
        tprs.append(tpr[-1] if fpr[-1] < target else float(np.interp(target, fpr, tpr)))
    return (float(np.mean(aucs)) if aucs else float('nan'),
            float(np.mean(aups)) if aups else float('nan'),
            float(np.mean(tprs)) if tprs else float('nan'))

4. Input Binding Helpers (support ANN, CNN, RNN models)

In [16]:
def get_shape(t):
    """Return input tensor shape as tuple."""
    try:
        return t.shape.as_list()
    except AttributeError:
        return tuple(t.shape)

def make_row_sequence(images: np.ndarray) -> np.ndarray:
    """
    Convert images (N,H,W,3) into row-wise sequences (N, H, W*3).
    For H=W=50 -> (N, 50, 150).
    """
    if images.ndim != 4 or images.shape[-1] != 3:
        raise ValueError(f"Expected images (N,H,W,3), got {images.shape}")
    N, H, W, C = images.shape
    return images.reshape(N, H, W*C)

def build_inputs_by_type(images: np.ndarray, is_night: np.ndarray, model: tf.keras.Model):
    """
    Bind inputs depending on model type:
    - ANN: flattened image (+ isNight)
    - CNN: image tensor (N,H,W,3) + isNight
    - RNN: row sequence (N,H,W*3) + isNight
    """
    img_tensor = images.astype(np.float32)
    meta_tensor = is_night.reshape(-1,1).astype(np.float32)

    # Flattened version for ANN
    flat_tensor = img_tensor.reshape(img_tensor.shape[0], -1)
    flat_plus_meta = np.concatenate([flat_tensor, meta_tensor], axis=1)

    # Row-sequence for RNN
    ranks = [len(get_shape(t)) for t in model.inputs]
    need_seq = any(r == 3 for r in ranks)
    seq_tensor = make_row_sequence(img_tensor) if need_seq else None

    if BIND_BY_NAME:
        feed = {}
        for t in model.inputs:
            name  = t.name.split(':')[0]
            shape = get_shape(t)
            rank  = len(shape)

            if rank == 4:
                feed[name] = img_tensor
            elif rank == 3:
                feed[name] = seq_tensor
            elif rank == 2:
                if shape[-1] in (1, None):
                    feed[name] = meta_tensor
                else:
                    feed[name] = flat_plus_meta
            else:
                feed[name] = img_tensor
        return feed
    else:
        feeds = []
        for t in model.inputs:
            shape = get_shape(t)
            rank  = len(shape)
            if rank == 4:
                feeds.append(img_tensor)
            elif rank == 3:
                feeds.append(seq_tensor)
            elif rank == 2:
                if shape[-1] in (1, None):
                    feeds.append(meta_tensor)
                else:
                    feeds.append(flat_plus_meta)
            else:
                feeds.append(img_tensor)
        return feeds

# Per-class TP/FP/FN/TN extraction from confusion matrix 
def per_class_confusion(cm: np.ndarray) -> pd.DataFrame:
    """Return per-class TP, FP, FN, TN as a DataFrame for detailed analysis."""
    K = cm.shape[0]
    total = cm.sum()
    rows = []
    for k in range(K):
        TP = cm[k,k]
        FN = cm[k,:].sum() - TP
        FP = cm[:,k].sum() - TP
        TN = total - (TP + FP + FN)
        rows.append({'class_id': k, 'class_name': CLASS_NAMES[k],
                     'TP': int(TP), 'FP': int(FP), 'FN': int(FN), 'TN': int(TN)})
    return pd.DataFrame(rows)

5. Dataset Builders: Clean (by split) and Adversarial

In [17]:
def build_clean(df_path: Path, split: str) -> pd.DataFrame:
    """
    Load the original (clean) annotations CSV and filter by split ('train' / 'val' / 'test').

    Args:
        df_path: Path to the original annotations CSV (e.g., new_annotations.csv).
        split:   Which split to keep ('train', 'val', or 'test').

    Returns:
        A DataFrame containing only rows of the requested split, with an 'isNight' column ensured.

    Raises:
        ValueError: If the 'split' column is missing or no rows match the requested split.
    """
    """Return only the requested clean split (we will pass split='test')."""
    df = load_csv_safely(df_path)
    if 'split' not in df.columns:
        raise ValueError("Clean CSV missing 'split' column")
    
    out = df[df['split'].astype(str).str.lower()==split.lower()].copy()

    # Ensure 'isNight' exists (default 0 if missing)
    if 'isNight' not in out.columns:
        out['isNight'] = 0

    if out.empty:
        raise ValueError(f"No rows for split={split}")
    return out

def build_adv(df_path: Path, enforce_test: bool = True) -> pd.DataFrame:
    """
    Load an adversarial annotations CSV (Annotations_*.csv).

    The file must contain at least 'image_id' and 'label'.
    If 'isNight' is missing, default it to 0.

    Args:
        df_path: Path to the adversarial annotations CSV.

    Returns:
        A DataFrame ready for evaluation (image_id, label, isNight, ...).

    Raises:
        ValueError: If required columns are missing.
    """
    """
    Load adversarial annotations and (optionally) keep only test split if a 'split' column exists.
    """
    df = load_csv_safely(df_path)
    if 'image_id' not in df.columns or 'label' not in df.columns:
        raise ValueError(f"Adversarial CSV missing columns: {df_path}")
    
    if enforce_test and 'split' in df.columns:
        df = df[df['split'].astype(str).str.lower() == 'test'].copy()
        if df.empty:
            raise ValueError(f"{df_path.name}: no rows with split='test'")

    if 'isNight' not in df.columns:
        df['isNight'] = 0

    return df

def get_dataset_df(name: str) -> pd.DataFrame:
    """
    Resolve a dataset name (key in DATASETS) to a DataFrame:
      - If it's a clean split (e.g., 'clean_test'), call build_clean(...).
      - Otherwise, treat it as adversarial CSV and call build_adv(...).

    Args:
        name: Dataset key from DATASETS dict.

    Returns:
        DataFrame for the requested dataset.
    """
    src = DATASETS[name]
    if name in SPLIT_FOR_CLEAN:
        return build_clean(src, split=SPLIT_FOR_CLEAN[name])
    return build_adv(src, enforce_test=True)

6. Load Models & Prepare State

In [18]:
# Load trained models (disable compilation for evaluation-only mode) 
MODELS = {
    'ANN': tf.keras.models.load_model(ANN_PATH, compile=False),
    'CNN': tf.keras.models.load_model(CNN_PATH, compile=False),
    'RNN': tf.keras.models.load_model(RNN_PATH, compile=False),
}

# Print input tensor names for each model (useful for debugging input binding)
print({k:[t.name for t in v.inputs] for k,v in MODELS.items()})

# Accumulators for evaluation results 
ACCUM_SUMMARIES = []    # stores per-run summary metrics (accuracy, F1, AUROC, etc.)
ACCUM_TPFP = []         # stores detailed per-class TP/FP/FN/TN tables
ACCUM_CMS = []          # stores confusion matrices

{'ANN': ['input_1'], 'CNN': ['image', 'isNight'], 'RNN': ['row_sequence', 'isNight']}


7. valuation Function: Run Model on Dataset and Collect Metrics

In [19]:
def evaluate_one(model_key: str, dataset_key: str):
    """
    Evaluate a given model on a given dataset and print/save metrics.

    Args:
        model_key:   'ANN', 'CNN', or 'RNN' (key in MODELS).
        dataset_key: key from DATASETS dict (e.g., 'clean_test', 'Annotations_ANN_fgsm').

    Workflow:
        - Load dataset and normalize labels
        - Run model predictions in batches
        - Compute probabilities, predictions, confusion matrix
        - Print confusion matrix, classification report, per-class TP/FP/FN/TN
        - Compute global metrics (accuracy, precision, recall, F1, AUROC, AUPRC, ECE)
        - Store results in accumulators for later export (Excel)
    """
    model = MODELS[model_key]
    df = get_dataset_df(dataset_key)

    # Safety: ensure we only evaluate on test if split column exists
    if 'split' in df.columns:
        unique_splits = sorted(df['split'].astype(str).str.lower().unique().tolist())
        if unique_splits != ['test']:
            raise ValueError(f"{dataset_key}: expected only 'test' split, found {unique_splits}")
        
    # Prepare inputs 
    paths = [resolve_image_path(p) for p in df['image_id'].astype(str).tolist()]
    y_true = normalize_labels(df['label'])
    isN    = df['isNight'].astype(int).values

    probs_all, preds_all = [], []
    idxs = np.arange(len(paths))
    t0 = time.time()

    # Batch prediction loop 
    for s in range(0, len(idxs), BATCH_SIZE):
        part = idxs[s:s+BATCH_SIZE]
        imgs = np.stack([read_image_tensor(paths[i]) for i in part], axis=0)
        meta = isN[part]

        # Bind inputs depending on model type
        X = build_inputs_by_type(imgs, meta, model)

        # Forward pass
        logits = model(X, training=False)
        if isinstance(logits,(list,tuple)): logits = logits[0]

        # Softmax probabilities and argmax predictions
        probs = tf.nn.softmax(logits, axis=-1).numpy()
        preds = probs.argmax(axis=1)

        probs_all.append(probs); 
        preds_all.append(preds)

    elapsed = time.time() - t0

    # Concatenate batch results
    probs_all = np.concatenate(probs_all, axis=0)
    preds_all = np.concatenate(preds_all, axis=0)

    # Confusion matrix and reports 
    cm = confusion_matrix(y_true, preds_all, labels=[0,1,2])

    # Save confusion matrix in long-form (flattened) for plotting later
    cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
    cm_long = cm_df.reset_index().melt(id_vars='index', var_name='pred', value_name='count')
    cm_long = cm_long.rename(columns={'index': 'true'})
    cm_long.insert(0, 'dataset', dataset_key)
    cm_long.insert(0, 'model', model_key)
    ACCUM_CMS.append(cm_long)

    print(f"\n=== {model_key} @ {dataset_key} ===")
    print("Confusion matrix (rows=True, cols=Pred):")
    print(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES))

    print("\nClassification report:")
    print(classification_report(y_true, preds_all, target_names=CLASS_NAMES, digits=4))

    tpfp = per_class_confusion(cm)
    print("\nTP/FP/FN/TN per class:")
    print(tpfp)

    # Global metrics
    acc = accuracy_score(y_true, preds_all)
    prec_macro  = precision_score(y_true, preds_all, average='macro', zero_division=0)
    rec_macro   = recall_score(y_true, preds_all, average='macro', zero_division=0)
    f1_macro    = f1_score(y_true, preds_all, average='macro', zero_division=0)

    prec_weight = precision_score(y_true, preds_all, average='weighted', zero_division=0)
    rec_weight  = recall_score(y_true, preds_all, average='weighted', zero_division=0)
    f1_weight   = f1_score(y_true, preds_all, average='weighted', zero_division=0)

    auroc_macro, auprc_macro, tpr1 = roc_ovr_metrics(y_true, probs_all)
    ece = ece_top1(probs_all, y_true, n_bins=NUM_BINS_ECE)

    ms_per_sample = 1000.0 * elapsed / len(y_true)

    # Save results into accumulators 
    summary = {
        'model': model_key, 'dataset': dataset_key, 'n_samples': int(len(y_true)),
        'acc': acc, 'precision_macro': prec_macro, 'recall_macro': rec_macro, 'f1_macro': f1_macro,
        'precision_weighted': prec_weight, 'recall_weighted': rec_weight, 'f1_weighted': f1_weight,
        'auroc_macro_ovr': auroc_macro, 'auprc_macro_ovr': auprc_macro, 'tpr_at_fpr_0.01_macro_ovr': tpr1,
        'ece_top1': ece, 'ms_per_sample': ms_per_sample,
        'color_space': COLOR_SPACE, 'bind_by_name': BIND_BY_NAME
    }
    
    tpfp_long = tpfp.copy()
    tpfp_long.insert(0, 'dataset', dataset_key)
    tpfp_long.insert(0, 'model', model_key)

    ACCUM_SUMMARIES.append(summary)
    ACCUM_TPFP.append(tpfp_long)

print("Call evaluate_one('CNN','clean_test') etc.")

Call evaluate_one('CNN','clean_test') etc.


8. Export Results to Excel (Summary + Per-class TP/FP/FN/TN)

In [20]:
# Writes two sheets: 'Summary' and 'TPFPFNTN'
# Tries 'openpyxl' first; falls back to 'xlsxwriter' if available

def save_summary_xlsx(path: Path = None):
    """
    Save accumulated evaluation results into a single Excel workbook:
      - Sheet 'Summary'   : one row per (model, dataset) with global metrics
      - Sheet 'TPFPFNTN' : detailed per-class TP/FP/FN/TN for each (model, dataset)

    Args:
        path: Optional Path to the output .xlsx file. Defaults to OUT_XLSX.

    Returns:
        The output path if saved, else None.
    """
    if path is None: path = OUT_XLSX

    if not ACCUM_SUMMARIES:
        print("Nothing to save yet. Run evaluate_one(...) first.")
        return None
    
    # Build DataFrames from accumulators
    df_summary = pd.DataFrame(ACCUM_SUMMARIES)
    df_tpfp = pd.concat(ACCUM_TPFP, ignore_index=True)
    df_cm      = pd.concat(ACCUM_CMS, ignore_index=True) if ACCUM_CMS else pd.DataFrame()

    # Write the workbook
    with pd.ExcelWriter(path, engine='xlsxwriter') as w:
        df_summary.to_excel(w, index=False, sheet_name='Summary')
        df_tpfp.to_excel(w, index=False, sheet_name='TPFPFNTN')
        if not df_cm.empty:
            df_cm.to_excel(w, index=False, sheet_name='Confusions')
            
    print("✓ Saved:", path)
    return path

print("When ready, run: save_summary_xlsx()")

When ready, run: save_summary_xlsx()


9. Sanity Check on Training Data

In [21]:
# Quickly verifies that preprocessing + input binding are consistent with how the model was trained.
# If accuracy here is low, something is wrong in preprocessing.

def sanity_check(model_key: str, n_samples: int = 128):
    """
    Run a small sanity check on the training set.

    Args:
        model_key:   'ANN', 'CNN', or 'RNN' (which model to test).
        n_samples:   Number of samples to randomly draw from the train split.

    Returns:
        acc: Accuracy on the selected training samples.
    """
    model = MODELS[model_key]

    # Load subset of training data
    df = get_dataset_df('clean_train')
    if len(df) > n_samples:
        df = df.sample(n_samples, random_state=42)

    # Prepare inputs
    paths = [resolve_image_path(p) for p in df['image_id'].astype(str).tolist()]
    y_true = normalize_labels(df['label'])
    isN    = df['isNight'].astype(int).values
    imgs = np.stack([read_image_tensor(p) for p in paths], axis=0)

    # Bind inputs depending on model type
    X = build_inputs_by_type(imgs, isN, model)

    # Forward pass
    logits = model(X, training=False)
    if isinstance(logits,(list,tuple)): logits = logits[0]

    # Predictions
    probs = tf.nn.softmax(logits, axis=-1).numpy()
    preds = probs.argmax(axis=1)

    # Accuracy on train subset
    acc = accuracy_score(y_true, preds)
    print(f"Sanity on TRAIN ({len(df)} samples) — {model_key}: accuracy = {acc:.4f}")
    return acc

print("Run sanity_check('CNN') / ('ANN') / ('RNN')")

Run sanity_check('CNN') / ('ANN') / ('RNN')


10. Sanity Checks, Clean Evaluations, and Save to Excel

In [ ]:
# Show which dataset keys are configured
print("Datasets available:", list(DATASETS.keys()))

# Sanity checks on a small TRAIN subset
# If preprocessing/binding matches training, accuracy should be high.
try:
    sanity_check('ANN')
    sanity_check('CNN')
    sanity_check('RNN')
except Exception as e:
    print("Sanity check error:", e)

# Minimal evaluations on CLEAN test set for all three models
# Prints confusion matrix, classification report, TP/FP/FN/TN per class,
# and logs summary metrics into the accumulators.
try:
    evaluate_one('ANN','clean_test')
    evaluate_one('ANN','Annotations_ANN_fgsm')
    evaluate_one('ANN','Annotations_ANN_pgd')
    evaluate_one('ANN','Annotations_CNN_fgsm')
    evaluate_one('ANN','Annotations_CNN_pgd')
    evaluate_one('ANN','Annotations_RNN_fgsm')
    evaluate_one('ANN','Annotations_RNN_pgd')

    evaluate_one('CNN','clean_test')
    evaluate_one('CNN','Annotations_ANN_fgsm')
    evaluate_one('CNN','Annotations_ANN_pgd')
    evaluate_one('CNN','Annotations_CNN_fgsm')
    evaluate_one('CNN','Annotations_CNN_pgd')
    evaluate_one('CNN','Annotations_RNN_fgsm')
    evaluate_one('CNN','Annotations_RNN_pgd')
 
    evaluate_one('RNN','clean_test')
    evaluate_one('RNN','Annotations_ANN_fgsm')
    evaluate_one('RNN','Annotations_ANN_pgd')
    evaluate_one('RNN','Annotations_CNN_fgsm')
    evaluate_one('RNN','Annotations_CNN_pgd')
    evaluate_one('RNN','Annotations_RNN_fgsm')
    evaluate_one('RNN','Annotations_RNN_pgd')

except Exception as e:
    print("Evaluation error:", e)

# Save current accumulated results to Excel
# Writes two sheets: 'Summary' (global metrics) and 'TPFPFNTN' (per-class).
try:
    save_summary_xlsx(Path("../../eval_tables/evaluation_summary.xlsx"))
except Exception as e:
    print("Save error:", e)

Datasets available: ['clean_train', 'clean_test', 'Annotations_ANN_fgsm', 'Annotations_ANN_pgd', 'Annotations_CNN_fgsm', 'Annotations_CNN_pgd', 'Annotations_RNN_fgsm', 'Annotations_RNN_pgd']
Sanity on TRAIN (128 samples) — ANN: accuracy = 0.9922
Sanity on TRAIN (128 samples) — CNN: accuracy = 1.0000
Sanity on TRAIN (128 samples) — RNN: accuracy = 0.9922

=== ANN @ clean_test ===
Confusion matrix (rows=True, cols=Pred):
           go  warning  stop
go       7213        5    11
warning     0      493     6
stop        1        3  6701

Classification report:
              precision    recall  f1-score   support

          go     0.9999    0.9978    0.9988      7229
     warning     0.9840    0.9880    0.9860       499
        stop     0.9975    0.9994    0.9984      6705

    accuracy                         0.9982     14433
   macro avg     0.9938    0.9951    0.9944     14433
weighted avg     0.9982    0.9982    0.9982     14433


TP/FP/FN/TN per class:
   class_id class_name    TP  FP